# 💻 Hands-On Lab Part 3 — Index Exploration vs. Radiative Transfer Models (45 Minutes)

In this final phase of the workshop, you will step away from guided scripts to conduct an independent algorithmic comparison. You will choose an advanced index, calculate its values across our farm's Area of Interest (AOI), and try to interpret the results.

---

## 🔍 Step 1: Researching Alternative Indices

https://clearsky.vision/knowledge/sentinel2-indices-cheatsheet

---

## 🧪 Step 2: Student Challenge — Calculate Your Chosen Index

Using your chosen index from the matrix above, write the missing array algebra inside the execution block below. 

> ⚠️ **Compilation Rule:** Remember that different Sentinel-2 bands have different native pixel sizes. If your chosen index mixes $10\text{m}$ bands (like `B02`, `B04`, `B08`) with $20\text{m}$ bands (like `B11` or `B12`), you **must** use the `.rio.reproject_match()` protocol you learned in Part 2!

* **Action Required:** Complete the code snippet below for your selected index and plot the results.

In [1]:
# Uncomment the line below if you need to install the dependencies in your environment
# !pip install pystac-client pystac shapely planetary-computer requests matplotlib

import json
from pystac_client import Client
import planetary_computer as pc
from shapely.geometry import shape, mapping
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
import rioxarray



print("📦 Libraries imported successfully! Ready to connect to the datacube.")

# Define a bounding box around a high-production agricultural region
# Format: [min_longitude, min_latitude, max_longitude, max_latitude]
aoi_bbox = [13.15, 52.35, 13.18, 52.38] 

# Convert the bounding box into a standard GeoJSON geometry dictionary
aoi_geometry = {
    "type": "Polygon",
    "coordinates": [[
        [aoi_bbox[0], aoi_bbox[1]],
        [aoi_bbox[0], aoi_bbox[3]],
        [aoi_bbox[2], aoi_bbox[3]],
        [aoi_bbox[2], aoi_bbox[1]],
        [aoi_bbox[0], aoi_bbox[1]]
    ]]
}

print("Farm boundary geometry locked in. Ready to query.")


# Connect to the global cloud catalog endpoint
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

# Execute the programmatic search query
search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=aoi_geometry,
    datetime="2025-04-01/2025-04-10", # Target spring growing season
    query={"eo:cloud_cover": {"lt": 10}} # Filter: less than 10% cloud cover
)

# Fetch all matching items found in the cloud registry
items = list(search.get_items())
print(f"📡 API Query Complete! Found {len(items)} cloud-free satellite scenes matching your criteria.")

s2_items = [pc.sign(item) for item in search.get_items()]
cropping_poly = shape(aoi_geometry)

print(f"📈 Analytics engine armed. Ready to process {len(s2_items)} multi-spectral scenes.")


Bad key keymap.all_axes in file matplotlibrc, line 398 ('keymap.all_axes : a                 # enable all axes')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.10.9/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution


📦 Libraries imported successfully! Ready to connect to the datacube.
Farm boundary geometry locked in. Ready to query.


/root/.local/lib/python3.12/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


📡 API Query Complete! Found 4 cloud-free satellite scenes matching your criteria.
📈 Analytics engine armed. Ready to process 4 multi-spectral scenes.


In [ ]:
def process_student_challenge_index(index_frame):
    if not s2_items:
        print("❌ No imagery layers found.")
        return
        
    current_item = s2_items[index_frame]
    
    try:
        # 📥 1. STREAM AND CLIP YOUR SPECIFIC BANDS
        # (Example below sets up basic bands; modify according to your chosen index)
        url_b08 = current_item.assets["B08"].href # NIR
        raster_b08 = rioxarray.open_rasterio(url_b08)
        clip_b08 = raster_b08.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True)
        nir = clip_nir = clip_b08.data[0].astype(float)
        
        url_b04 = current_item.assets["B04"].href # Red
        raster_b04 = rioxarray.open_rasterio(url_b04)
        clip_b04 = raster_b04.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True)
        red = clip_b04.data[0].astype(float)
        
        # NOTE: If you use a 20m band (like B11 for NDWI), uncomment and use this alignment block:
        # url_b11 = current_item.assets["B11"].href
        # raster_b11 = rioxarray.open_rasterio(url_b11)
        # clip_b11 = raster_b11.rio.clip([cropping_poly], crs="EPSG:4326", from_disk=True).rio.reproject_match(clip_b08)
        # swir = clip_b11.data[0].astype(float)

        # 🧮 2. WRITE YOUR SPECIFIC MATRIX ARITHMETIC HERE
        # ----------------------------------------------------
        # STUDENT TODO EXAMPLES:
        # For SAVI:  calculated_index = ((nir - red) / (nir + red + 0.5)) * 1.5
        # For NDWI:  calculated_index = (nir - swir) / (nir + swir + 1e-5)
        
        calculated_index = (nir - red) / (nir + red + 1e-5) # Placeholder default (NDVI)
        # ----------------------------------------------------
        
        # Render the custom output map
        plt.figure(figsize=(10, 8))
        img_plot = plt.imshow(calculated_index, cmap="viridis")
        plt.colorbar(img_plot, label="Custom Index Value")
        plt.title(f"🚀 Custom Research Index Assessment\n📅 Date: {str(current_item.datetime)[:16]}", fontsize=12, fontweight='bold')
        plt.axis("off")
        plt.show()
        
    except Exception as e:
        print(f"💥 Compilation error in your index math: {e}")

# Slider to scrub your custom calculation over time
if s2_items:
    widgets.interact(process_student_challenge_index, index_frame=widgets.IntSlider(min=0, max=len(s2_items)-1, step=1, value=0, description='Timeline:'))

interactive(children=(IntSlider(value=0, description='Timeline:', max=3), Output()), _dom_classes=('widget-int…

## 🤖 Step 3: Comparing Empirical Indices vs. Radiative Transfer Models



## 📝 Lab Assignment Report Questions

